## Applying Sentiment Analysis to the "Sentiment Analysis for Mental Health" Dataset

In [1]:
# Step 1: Load the Dataset
import pandas as pd

# Load the dataset
df = pd.read_csv(r'/content/Combined Data.csv',index_col=0)

# Display the first few rows of the dataset
print(df.head())

                                           statement   status
0                                         oh my gosh  Anxiety
1  trouble sleeping, confused mind, restless hear...  Anxiety
2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3  I've shifted my focus to something else but I'...  Anxiety
4  I'm restless and restless, it's been a month n...  Anxiety


In [2]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')

# Example of text cleaning function
def preprocess_text(text):
    # Convert any non-string input to string before processing
    text = str(text) if not isinstance(text, str) else text
    text = re.sub(r'\W', ' ', text)  # Remove non-alphanumeric characters
    text = text.lower()  # Convert to lowercase
    words = word_tokenize(text)  # Tokenize the text
    #words = [word for word in words if word not in stop_words]  # Remove stopwords
    return ' '.join(words)

# Apply the text preprocessing function
df['cleaned_text'] = df['statement'].apply(preprocess_text)
df[['cleaned_text','status']]

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,cleaned_text,status
0,oh my gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,all wrong back off dear forward doubt stay in ...,Anxiety
3,i ve shifted my focus to something else but i ...,Anxiety
4,i m restless and restless it s been a month no...,Anxiety
...,...,...
53038,nobody takes me seriously i ve 24m dealt with ...,Anxiety
53039,selfishness i don t feel very good it s like i...,Anxiety
53040,is there any way to sleep better i can t sleep...,Anxiety
53041,public speaking tips hi all i have to give a p...,Anxiety


In [3]:
# Step 3: Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_text'])
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3094042 stored elements and shape (53044, 5000)>

In [4]:
# Step 4: Model Training
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Drop rows where 'status' is NaN before splitting
df_cleaned = df.dropna(subset=['status'])

# Convert text to TF-IDF features for the cleaned DataFrame
# Ensure X is re-created based on the cleaned data if rows were dropped
# To maintain consistency, re-vectorize if rows are dropped from df.
# However, if 'status' NaNs are the ONLY problem, X is already fine if it maps to original df.
# Let's re-align X with df_cleaned for robustness.
vectorizer_cleaned = TfidfVectorizer(max_features=5000)
X_cleaned = vectorizer_cleaned.fit_transform(df_cleaned['cleaned_text'])

# Assuming 'status' is the label column
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, df_cleaned['status'], test_size=0.2, random_state=42)

# Train a logistic regression model
model = LogisticRegression(max_iter=1000) # Increased max_iter for convergence with potentially complex data
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                      precision    recall  f1-score   support

             Anxiety       0.80      0.77      0.78       777
             Bipolar       0.88      0.69      0.78       588
          Depression       0.71      0.74      0.72      3041
              Normal       0.86      0.96      0.91      3348
Personality disorder       0.63      0.49      0.55       234
              Stress       0.72      0.45      0.55       533
            Suicidal       0.71      0.67      0.69      2088

            accuracy                           0.77     10609
           macro avg       0.76      0.68      0.71     10609
        weighted avg       0.77      0.77      0.77     10609



In [6]:
# Step 5: Sentiment Prediction (Apply the trained model to new or unseen data to predict sentiment.)

# Define a new text statement for prediction
Experience = "I am feeling Low Nowadays"

# Preprocess the new text using the same function used for training data
cleaned_new_text = preprocess_text(Experience)

# Transform the cleaned new text into TF-IDF features using the fitted vectorizer
# The vectorizer expects an iterable, so we put the text in a list
new_text_vectorized = vectorizer_cleaned.transform([cleaned_new_text])

# Predict the sentiment of the new text
predicted_sentiment = model.predict(new_text_vectorized)

print(f"Experience: '{Experience}'")
print(f"Analysis: {predicted_sentiment[0]}")

Experience: 'I am feeling Low Nowadays'
Analysis: Depression
